In [11]:
import torch
import torch.nn as nn
import torch.optim as optim
import random
import numpy as np
import nltk
from nltk import word_tokenize
from datasets import Dataset, DatasetDict #, load_metric
# import torchtext
import tqdm
import evaluate
nltk.download('punkt')

[nltk_data] Downloading package punkt to /home/elena/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [12]:
from datasets import load_from_disk

In [13]:
seed = 1234

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.backends.cudnn.deterministic = True

In [15]:
ds = load_from_disk("/home/elena/emcomm/emcomm_captions/checkpoints/winoground_100_eval_best_max_val_10_len_99th_epoch_orig/messages/coco_val_message_captions_3_distractors")

ds

Dataset({
    features: ['coco_url', 'captions', 'image_id', 'features', 'message', 'message_truncated'],
    num_rows: 5000
})

In [17]:
src_sentences, trg_sentences = [], []
for row in ds:
    trg_sentences.append(row['captions'][0])
    src_sentences.append(row['message_truncated'])

In [18]:
src_sentences[:5], trg_sentences[:5]

([[24, 24, 4, 68, 10, 24, 10, 68, 4, 8],
  [30, 24, 2, 68, 4, 8, 2, 24, 4, 68],
  [10, 8, 24, 10, 2, 68, 19, 2, 8, 24],
  [2, 68, 10, 10, 68, 24, 48, 24, 8, 48],
  [24, 2, 68, 4, 8, 24, 24, 48, 68, 22]],
 ['A man is in a kitchen making pizzas.',
  'The dining table near the kitchen has a bowl of fruit on it.',
  'a person with a shopping cart on a city street ',
  'A person on a skateboard and bike at a skate park.',
  'a blue bike parked on a side walk '])

In [21]:
src, trg = [], []

for src_sent, trg_sent in zip(src_sentences, trg_sentences):
    src.append(word_tokenize(' '.join(str(s) for s in src_sent)))
    trg.append(word_tokenize(trg_sent))

print(src[:5])
print(trg[:5])

[['24', '24', '4', '68', '10', '24', '10', '68', '4', '8'], ['30', '24', '2', '68', '4', '8', '2', '24', '4', '68'], ['10', '8', '24', '10', '2', '68', '19', '2', '8', '24'], ['2', '68', '10', '10', '68', '24', '48', '24', '8', '48'], ['24', '2', '68', '4', '8', '24', '24', '48', '68', '22']]
[['A', 'man', 'is', 'in', 'a', 'kitchen', 'making', 'pizzas', '.'], ['The', 'dining', 'table', 'near', 'the', 'kitchen', 'has', 'a', 'bowl', 'of', 'fruit', 'on', 'it', '.'], ['a', 'person', 'with', 'a', 'shopping', 'cart', 'on', 'a', 'city', 'street'], ['A', 'person', 'on', 'a', 'skateboard', 'and', 'bike', 'at', 'a', 'skate', 'park', '.'], ['a', 'blue', 'bike', 'parked', 'on', 'a', 'side', 'walk']]


In [22]:
dataset = Dataset.from_dict({'src': src, 'trg': trg})

In [23]:
train_testvalid = dataset.train_test_split(test_size=0.2)
# Split the 10% test + valid in half test, half valid
test_valid = train_testvalid['test'].train_test_split(test_size=0.7)
# gather everyone if you want to have a single DatasetDict
data = DatasetDict({
    'train': train_testvalid['train'],
    'validation': test_valid['train'],
    'test': test_valid['test']})

In [24]:
data

DatasetDict({
    train: Dataset({
        features: ['src', 'trg'],
        num_rows: 4000
    })
    validation: Dataset({
        features: ['src', 'trg'],
        num_rows: 300
    })
    test: Dataset({
        features: ['src', 'trg'],
        num_rows: 700
    })
})

In [25]:
def preprocess_example(
    example,
    max_length,
    lower,
    sos_token,
    eos_token
):
    src_tokens = example["src"][:max_length]
    trg_tokens = example["trg"][:max_length]
    if lower:
        src_tokens = [token.lower() for token in src_tokens]
        trg_tokens = [token.lower() for token in trg_tokens]
    src_tokens = [sos_token] + src_tokens + [eos_token]
    trg_tokens = [sos_token] + trg_tokens + [eos_token]
    return {"src": src_tokens, "trg": trg_tokens}

In [26]:
max_length = 1_000
lower = True
sos_token = "<sos>"
eos_token = "<eos>"

fn_kwargs = {
    "max_length": max_length,
    "lower": lower,
    "sos_token": sos_token,
    "eos_token": eos_token,
}


data = data.map(preprocess_example, fn_kwargs=fn_kwargs)

Map: 100%|██████████| 700/700 [00:00<00:00, 10761.84 examples/s]


In [27]:
data['train'][0]

{'src': ['<sos>',
  '24',
  '10',
  '24',
  '8',
  '24',
  '10',
  '68',
  '19',
  '2',
  '8',
  '<eos>'],
 'trg': ['<sos>',
  'a',
  'group',
  'of',
  'people',
  'gathered',
  'around',
  'a',
  'table',
  'filled',
  'with',
  'food',
  '.',
  '<eos>']}

In [28]:
min_freq = 2
unk_token = "<unk>"
pad_token = "<pad>"

special_tokens = [
    unk_token,
    pad_token,
    sos_token,
    eos_token,
]

from collections import Counter

def build_vocab(iterator, min_freq=1, specials=None):
    counter = Counter()
    for seq in iterator:
        counter.update(seq)
    
    # Keep only tokens with frequency >= min_freq
    tokens = [tok for tok, freq in counter.items() if freq >= min_freq]
    
    # Add special tokens at the beginning
    if specials:
        tokens = specials + tokens
    
    # Create a mapping: token -> index
    vocab = {tok: idx for idx, tok in enumerate(tokens)}
    return vocab

# Example usage
# special_tokens = ["<pad>", "<unk>", "<bos>", "<eos>"]
# min_freq = 2

src_vocab = build_vocab(data['train']["src"], min_freq=min_freq, specials=special_tokens)
trg_vocab = build_vocab(data['train']["trg"], min_freq=min_freq, specials=special_tokens)

In [33]:
print(list(src_vocab.items())[:5])
print(list(trg_vocab.items())[:5])

[('<unk>', 0), ('<pad>', 1), ('<sos>', 4), ('<eos>', 11), ('24', 5)]
[('<unk>', 0), ('<pad>', 1), ('<sos>', 4), ('<eos>', 16), ('a', 5)]


In [35]:
print("woman" in src_vocab, "kitchen" in src_vocab)
print("woman" in trg_vocab, "kitchen" in trg_vocab)

False False
True True


In [36]:
assert src_vocab[unk_token] == trg_vocab[unk_token]
assert src_vocab[pad_token] == trg_vocab[pad_token]

unk_index = src_vocab[unk_token]
pad_index = src_vocab[pad_token]